# Pipeline model

> Load weights, build the graph, compile it, run ``execute``.


In [ ]:
#| default_exp model


In [ ]:
#| hide
from nbdev.showdoc import *


## Two types, don't confuse them

| Name | What it is |
|------|------------|
| `SmiTedModel` (in `graph`) | The neural net `Module` |
| `SmiTedPipelineModel` (here) | MAX's pipeline wrapper around that net |

The pipeline model is glue: adapt weights → `build_graph` → `session.load` → on each request, forward the two buffers.


In [ ]:
#| export
from __future__ import annotations

import logging
import time
from dataclasses import dataclass
from typing import ClassVar

from max.driver import Buffer, Device
from max.engine import InferenceSession, Model
from max.graph.weights import Weights, WeightsAdapter
from max.nn.transformer import ReturnLogits
from max.pipelines.context import TextContext
from max.pipelines.lib import (
    KVCacheConfig,
    ModelInputs,
    ModelOutputs,
    PipelineConfig,
    PipelineModel,
)

from mat_gram01.batch_processor import SmiTedBatchProcessor
from mat_gram01.graph import build_graph
from mat_gram01.model_config import SmiTedModelConfig

logger = logging.getLogger("max.pipelines")


In [ ]:
#| export
@dataclass
class SmiTedInputs(ModelInputs):
    "Token ids and attention mask buffers for one encoder batch."
    next_tokens_batch: Buffer
    attention_mask: Buffer


## SmiTedPipelineModel


In [ ]:
#| export
class SmiTedPipelineModel(PipelineModel[TextContext]):
    "MAX pipeline model for SMI-TED embeddings generation."
    batch_processor_cls: ClassVar[type[SmiTedBatchProcessor]] = SmiTedBatchProcessor
    model_config_cls: ClassVar[type[SmiTedModelConfig]] = SmiTedModelConfig

    def __init__(
        self,
        pipeline_config: PipelineConfig,
        session: InferenceSession,
        devices: list[Device],
        kv_cache_config: KVCacheConfig,
        weights: Weights,
        adapter: WeightsAdapter | None = None,
        return_logits: ReturnLogits = ReturnLogits.ALL,
        max_batch_size: int = 1,
    ) -> None:
        super().__init__(
            pipeline_config,
            session,
            devices,
            kv_cache_config,
            weights,
            adapter,
            return_logits,
            max_batch_size=max_batch_size,
        )
        self.model = self.load_model(session)

    def execute(self, model_inputs: ModelInputs) -> ModelOutputs:
        assert isinstance(model_inputs, SmiTedInputs)
        model_outputs = self.model.execute(
            model_inputs.next_tokens_batch, model_inputs.attention_mask
        )
        assert self.batch_processor is not None
        return self.batch_processor.process_outputs(model_outputs)

    def load_model(self, session: InferenceSession) -> Model:
        logger.info("Building and compiling SMI-TED graph...")
        before = time.perf_counter()
        if self.adapter:
            state_dict = self.adapter(dict(self.weights.items()))
        else:
            state_dict = {
                key: value.data() for key, value in self.weights.items()
            }
        config = SmiTedModelConfig.initialize(self.pipeline_config)
        graph = build_graph(config, state_dict)
        after_build = time.perf_counter()
        logger.info("Building graph took %.6f seconds", after_build - before)

        before_compile = time.perf_counter()
        model = session.load(graph, weights_registry=state_dict)
        after = time.perf_counter()
        logger.info("Compiling model took %.6f seconds", after - before_compile)
        logger.info(
            "Building and compiling SMI-TED took %.6f seconds", after - before
        )
        return model


### Checkpoint

Narrate the request path: *SMILES → tokenize → pad/mask → execute → embedding.*


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
